# COMPASS Validation Guide (Annotated Datasets)

This notebook is a **didactic, mode-specific guide** for automated annotation validation in COMPASS.

Covered prediction types:

- binary classification
- multiclass classification
- univariate regression
- multivariate regression
- hierarchical mixed-task prediction

The shell commands in sections 3 to 7 use paths relative to `src/full_stack/backend/`; point `--results_dir` and `--output_dir` at your own runs. Section 9 shows `evaluate()`, the evaluation library that covers every task kind in one call; it finds its files by itself and runs offline on the bundled examples.

## 1. Validation Workflow (Common to All Modes)

Validation runs in two layers:

1. `run_validation_metrics.py`
   - computes core metrics + core plots
2. `detailed_analysis.py`
   - computes extended diagnostics + row-level audit artifacts

Alongside both, `evaluate()` (section 9) scores every task kind from one tidy table, per group, with bootstrap 95% confidence intervals, and draws a standard figure set.

Recommended review order:

1. annotation contract artifacts (`annotation_contract*`)
2. core metrics JSON (`*_metrics.json`)
3. mode-specific plots
4. detailed rows + detailed text reports

## 2. Inputs and Template Files

Required inputs:

- participant outputs in `results/participant_runs/`
- binary mode: `--targets_file` (JSON only)
- non-binary modes: `--annotations_json`

Template examples:

- `utils/validation/with_annotated_dataset/annotation_templates/examples/binary_targets_example.json`
- `utils/validation/with_annotated_dataset/annotation_templates/examples/multiclass_annotations_example.json`
- `utils/validation/with_annotated_dataset/annotation_templates/examples/regression_univariate_annotations_example.json`
- `utils/validation/with_annotated_dataset/annotation_templates/examples/regression_multivariate_annotations_example.json`
- `utils/validation/with_annotated_dataset/annotation_templates/examples/hierarchical_annotations_example.json`


## 3. Binary Classification Validation

### Automated contract checks

- binary annotation JSON parses correctly
- participant ID alignment (annotation ↔ participant output)
- valid binary truth labels (`CASE` / `CONTROL`)
- extractable binary prediction payload

### Automated outputs (core)

- `binary_metrics_integrated.json`
- `integrated_confusion_matrix.png`

### Automated outputs (detailed)

- `detailed_analysis.txt`
- `detailed_analysis_binary.json`
- binary diagnostics plots (composite vs correctness, calibration, verdict accuracy, iteration improvement; availability depends on data support)

### How to interpret

- start with confusion matrix and sensitivity/specificity
- use Brier/ECE + calibration for probability reliability
- inspect verdict/composite diagnostics for critic-quality alignment


In [ ]:
# Binary validation
!python utils/validation/with_annotated_dataset/run_validation_metrics.py \
    --results_dir ../results/participant_runs \
    --prediction_type binary \
    --targets_file ../data/__TARGETS__/binary_targets.json \
    --output_dir ../results/analysis/binary_confusion_matrix \
    --disorder_groups "MAJOR_DEPRESSIVE_DISORDER,ANXIETY_DISORDERS"

!python utils/validation/with_annotated_dataset/detailed_analysis.py \
    --results_dir ../results/participant_runs \
    --prediction_type binary \
    --targets_file ../data/__TARGETS__/binary_targets.json \
    --output_dir ../results/analysis/details \
    --disorder_groups "MAJOR_DEPRESSIVE_DISORDER,ANXIETY_DISORDERS"


## 4. Multiclass Classification Validation

### Automated contract checks

- class label presence per participant
- participant ID alignment
- valid prediction extraction for multiclass payload

### Automated outputs (core)

- `multiclass_metrics.json`
- `annotation_contract_multiclass.json`
- multiclass confusion matrix / per-class metrics / calibration plots

### Automated outputs (detailed)

- `detailed_analysis_multiclass.json`
- `detailed_analysis_multiclass.txt`
- `detailed_rows_multiclass.json`
- `detailed_annotation_contract_multiclass.json/.txt`
- extended diagnostics (top confusions, confidence/entropy, label distribution)

### How to interpret

- read macro F1 first, then per-class F1
- inspect top confusions for error structure
- use confidence diagnostics to detect overconfident mistakes


In [ ]:
# Multiclass validation
!python utils/validation/with_annotated_dataset/run_validation_metrics.py \
    --results_dir ../results/participant_runs \
    --prediction_type multiclass \
    --annotations_json ../data/__TARGETS__/annotated_targets.json \
    --output_dir ../results/analysis/multiclass

!python utils/validation/with_annotated_dataset/detailed_analysis.py \
    --results_dir ../results/participant_runs \
    --prediction_type multiclass \
    --annotations_json ../data/__TARGETS__/annotated_targets.json \
    --output_dir ../results/analysis/multiclass_details


## 5. Univariate Regression Validation

### Automated contract checks

- exactly one numeric target output per participant
- participant ID alignment
- extractable univariate regression prediction

### Automated outputs (core)

- `regression_univariate_metrics.json`
- `annotation_contract_regression_univariate.json`
- parity + error bar plots

### Automated outputs (detailed)

- `detailed_analysis_regression_univariate.json/.txt`
- `detailed_rows_regression_univariate.json`
- `detailed_annotation_contract_regression_univariate.json/.txt`
- residual distribution, residual-vs-true, top-error diagnostics

### How to interpret

- MAE/RMSE = scale of error
- R2 = explained variance quality
- residual diagnostics = bias/heteroscedasticity structure


In [ ]:
# Univariate regression validation
!python utils/validation/with_annotated_dataset/run_validation_metrics.py \
    --results_dir ../results/participant_runs \
    --prediction_type regression_univariate \
    --annotations_json ../data/__TARGETS__/annotated_targets.json \
    --output_dir ../results/analysis/univariate_regression

!python utils/validation/with_annotated_dataset/detailed_analysis.py \
    --results_dir ../results/participant_runs \
    --prediction_type regression_univariate \
    --annotations_json ../data/__TARGETS__/annotated_targets.json \
    --output_dir ../results/analysis/univariate_regression_details


## 6. Multivariate Regression Validation

### Automated contract checks

- two or more numeric outputs per participant
- participant ID alignment
- extractable multivariate regression prediction

### Automated outputs (core)

- `regression_multivariate_metrics.json`
- `annotation_contract_regression_multivariate.json`
- parity + error bar plots

### Automated outputs (detailed)

- `detailed_analysis_regression_multivariate.json/.txt`
- `detailed_rows_regression_multivariate.json`
- `detailed_annotation_contract_regression_multivariate.json/.txt`
- residual diagnostics + top absolute error ranking

### How to interpret

- inspect per-output metrics before macro summary
- use top-error ranking to identify problematic output dimensions


In [ ]:
# Multivariate regression validation
!python utils/validation/with_annotated_dataset/run_validation_metrics.py \
    --results_dir ../results/participant_runs \
    --prediction_type regression_multivariate \
    --annotations_json ../data/__TARGETS__/annotated_targets.json \
    --output_dir ../results/analysis/multivariate_regression

!python utils/validation/with_annotated_dataset/detailed_analysis.py \
    --results_dir ../results/participant_runs \
    --prediction_type regression_multivariate \
    --annotations_json ../data/__TARGETS__/annotated_targets.json \
    --output_dir ../results/analysis/multivariate_regression_details


## 7. Hierarchical Mixed-Task Validation

### Automated contract checks (strict)

- node payload validity per participant
- **cross-participant schema consistency**:
  - same node IDs
  - same node mode per node ID
  - same regression output-key set per regression node

If this consistency is violated, validation stops with an explicit schema mismatch error.

### Schema consistency checklist

1. every participant has the same node set
2. each node keeps the same mode in all rows
3. regression node output keys are identical across rows
4. classification nodes always include a label
5. regression nodes always include numeric values

### Automated outputs (core)

- `hierarchical_metrics.json`
- `annotation_contract_hierarchical.json`
- node score/support + node-type distribution plots

### Automated outputs (detailed)

- `detailed_analysis_hierarchical.json/.txt`
- `detailed_rows_hierarchical.json`
- `detailed_annotation_contract_hierarchical.json/.txt`
- node coverage + hierarchical metric heatmap diagnostics

### How to interpret

- inspect per-node metrics before macro score
- inspect coverage and support before concluding model quality


In [ ]:
# Hierarchical mixed-task validation
!python utils/validation/with_annotated_dataset/run_validation_metrics.py \
    --results_dir ../results/participant_runs \
    --prediction_type hierarchical \
    --annotations_json ../data/__TARGETS__/annotated_targets.json \
    --output_dir ../results/analysis/hierarchical

!python utils/validation/with_annotated_dataset/detailed_analysis.py \
    --results_dir ../results/participant_runs \
    --prediction_type hierarchical \
    --annotations_json ../data/__TARGETS__/annotated_targets.json \
    --output_dir ../results/analysis/hierarchical_details


## 8. Cross-Mode Artifact Inspection Helper

Use this helper to quickly inspect metrics, contract validity, and top annotation issues across generated outputs.


In [ ]:
# Quick artifact inspection helper (metrics + contract + detailed rows)
import json
from pathlib import Path

out_dir = Path('../results/analysis')

print('\n[METRICS]')
for p in sorted(out_dir.rglob('*_metrics.json'))[:50]:
    payload = json.loads(p.read_text())
    ptype = payload.get('prediction_type')
    n_rows = payload.get('n_rows')
    contract = payload.get('annotation_contract') if isinstance(payload, dict) else None
    valid = contract.get('n_valid_rows') if isinstance(contract, dict) else None
    total = contract.get('n_rows') if isinstance(contract, dict) else None
    if valid is not None:
        print(f'- {p.name}: type={ptype} rows={n_rows} contract={valid}/{total}')
    else:
        print(f'- {p.name}: type={ptype} rows={n_rows}')

print('\n[ANNOTATION CONTRACT ISSUES]')
for p in sorted(out_dir.rglob('annotation_contract_*.json'))[:50]:
    payload = json.loads(p.read_text())
    issues = payload.get('issue_counts') if isinstance(payload.get('issue_counts'), dict) else {}
    top = sorted(issues.items(), key=lambda kv: int(kv[1] or 0), reverse=True)[:5]
    top_str = ', '.join([f'{k}={v}' for k, v in top]) if top else 'none'
    print(f"- {p.name}: valid={payload.get('n_valid_rows')}/{payload.get('n_rows')}; top_issues={top_str}")

print('\n[DETAILED ROW PAYLOADS]')
for p in sorted(out_dir.rglob('detailed_rows_*.json'))[:50]:
    payload = json.loads(p.read_text())
    print(f"- {p.name}: n_rows={payload.get('n_rows')}")


## 9. The Evaluation Library: `evaluate()`

Next to the mode-specific scripts above, the package has one entry point for every task kind: `evaluate()`. It reads predictions in any common shape (COMPASS result folders, JSON/JSONL job records, tidy or wide CSV/TSV, DataFrames, plain dicts), joins them to annotations (the JSON templates, JSONL, or CSV/TSV with a column mapping) and normalizes both into **one tidy long table**: one row per prediction instance x node x output, with a `status`, so a missing prediction is counted instead of dropped.

From that table it scores every output and node per group (any column: tier, predictor, site, ...) with seeded bootstrap 95% confidence intervals that resample participants, and draws a standard figure set. Predictions are scored as given, never recalibrated.

| | Legacy scripts (sections 3 to 7) | `evaluate()` |
|---|---|---|
| Task kinds | one script call per prediction type | binary, multiclass, multi-label, regression, hierarchical and mixed trees in one call |
| Inputs | result folders + targets or annotations JSON | result folders, JSON/JSONL records, CSV/TSV, DataFrames, dicts |
| Metrics | one value per metric | per group, with 95% CI, `n`, `n_total`, `coverage` |
| Extras | detailed row audit | paired comparison of configurations, test-retest across repeats |

The cells below run offline on the bundled files in `annotation_templates/examples/` and write to a temporary folder. The examples hold three or four participants, so their intervals are wide or degenerate: they show the shape of the outputs, not a result. `README.md` in this folder documents every input shape, metric and figure.

In [ ]:
# Setup: find the library from wherever the notebook runs (repository root, backend, or this folder).
import subprocess
import sys
import tempfile
import warnings
from pathlib import Path

import pandas as pd

LIB_REL = Path('src/full_stack/backend/utils/validation/with_annotated_dataset')
here = Path.cwd().resolve()
roots = [p for p in (here, *here.parents) if (p / LIB_REL).is_dir()]
if not roots:
    raise RuntimeError('Run this notebook from inside the COMPASS repository.')
REPO = roots[0]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.full_stack.backend.utils.validation.with_annotated_dataset import evaluate

EXAMPLES = REPO / LIB_REL / 'annotation_templates' / 'examples'
OUT = Path(tempfile.mkdtemp(prefix='compass_evaluate_example_'))
# With three or four participants some bootstrap replicates miss a class; numpy then warns about empty means.
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)
print('examples:', EXAMPLES)
print('outputs: ', OUT)

### 9.1 Tidy input: predictions and annotations as tables

`predictions_tidy_example.csv` is a long table, one row per participant x output: grouping columns (`tier`, `predictor`, `repeat`), a binary `root` node (`pred_label`, `p_positive`) and a regression node `severity` with the output `total_score`. Participant 04 has no `total_score` prediction.

`annotations_table_example.csv` holds the truth in columns. `annotation_columns` maps each target to its column; the other columns (`site`, the subscales) become covariates that `group_by` can use.

In [ ]:
result = evaluate(
    EXAMPLES / 'predictions_tidy_example.csv',
    EXAMPLES / 'annotations_table_example.csv',
    id_column='participant_id',
    annotation_columns={'root': 'diagnosis', 'total_score': 'total_score'},
    group_by=['tier', 'predictor'],
    n_boot=500,
    seed=1,
    out_dir=OUT / 'tidy_example',
)
result.tidy[['participant_id', 'node_id', 'output', 'kind', 'status',
             'true_label', 'pred_label', 'p_positive', 'true_value', 'pred_value']]

### 9.2 Metrics

`result.metrics` is long: group x node x output x metric, with `value`, `ci_low`, `ci_high` and the counts `n` (scored), `n_total` and `coverage`. `result.metrics_wide` holds the same numbers as one row per group x node x output. The missing `total_score` prediction shows as `n` 3 of `n_total` 4 (coverage 0.75); classification nodes also report `accuracy_missing_as_wrong`.

In [ ]:
shown = ['accuracy', 'balanced_accuracy', 'auroc', 'brier', 'pearson_r', 'mae', 'bias']
columns = ['tier', 'predictor', 'node_id', 'output', 'metric', 'value', 'ci_low', 'ci_high', 'n', 'n_total', 'coverage']
result.metrics.loc[result.metrics['metric'].isin(shown), columns].round(3)

### 9.3 Job records and a paired comparison

Predictions can also be plain records, one per prediction job, the way a validation harness writes them: an id, grouping fields, a `prediction` block and `ok`. Here two predictors classify the participants of `multiclass_annotations_example.json`; one decision-model run failed (`ok: False`) and is counted as a missing prediction. `compare` scores both predictors on the participants they share and gives a paired bootstrap interval for the difference (`diff = b - a`).

In [ ]:
def job(pid, predictor, label=None, probs=None, ok=True):
    record = {'participant_id': pid, 'predictor': predictor, 'ok': ok}
    if ok:
        record['prediction'] = {'label': label, 'probs': probs}
    return record

records = [
    job('01', 'llm', 'subtype_a', {'subtype_a': 0.7, 'subtype_b': 0.2, 'subtype_c': 0.1}),
    job('02', 'llm', 'subtype_c', {'subtype_a': 0.2, 'subtype_b': 0.3, 'subtype_c': 0.5}),
    job('03', 'llm', 'subtype_c', {'subtype_a': 0.1, 'subtype_b': 0.2, 'subtype_c': 0.7}),
    job('01', 'decision', 'subtype_a', {'subtype_a': 0.6, 'subtype_b': 0.3, 'subtype_c': 0.1}),
    job('02', 'decision', 'subtype_b', {'subtype_a': 0.2, 'subtype_b': 0.6, 'subtype_c': 0.2}),
    job('03', 'decision', ok=False),
]
multiclass = evaluate(
    records,
    EXAMPLES / 'multiclass_annotations_example.json',
    group_by=['predictor'],
    compare=('predictor', 'llm', 'decision'),
    n_boot=500,
    seed=1,
    out_dir=OUT / 'multiclass_example',
)
metrics = multiclass.metrics
metrics.loc[metrics['metric'].isin(['accuracy', 'accuracy_missing_as_wrong', 'macro_f1']),
            ['predictor', 'metric', 'value', 'ci_low', 'ci_high', 'n', 'n_total', 'coverage']].round(3)

In [ ]:
multiclass.comparison.round(3)

### 9.4 Files and figures

With `out_dir`, every table is written as CSV (metric tables also as JSON), plus `summary.json` and a standard figure set in `figures/`: density scatter and residual plots per regression output, forest plots of the main metrics, metric-by-tier lines, confusion matrices, ROC and calibration curves, and paired-comparison plots. One colorblind-safe palette is used throughout; `figure_formats=('png', 'pdf')` adds vector copies.

In [ ]:
for name, path in sorted(result.files.items()):
    print(f'{name:<14} {Path(path).relative_to(OUT)}')
print()
for path in result.figures:
    print(Path(path).relative_to(OUT))

In [ ]:
try:
    from IPython.display import Image
except ImportError:  # plain Python: just name the file
    Image = None
scatter = next(p for p in result.figures if Path(p).name.startswith('scatter_'))
Image(filename=scatter) if Image else scatter

### 9.5 The same evaluation from the command line

From the repository root, `python -m src.full_stack.backend.utils.validation.with_annotated_dataset evaluate` takes the same options (`run_evaluation.py evaluate` in this folder does the same without importing the rest of the backend):

```bash
python -m src.full_stack.backend.utils.validation.with_annotated_dataset evaluate \
    --predictions results/participant_runs \
    --annotations annotations.json \
    --group-by tier,predictor \
    --out results/analysis/evaluation
```

After a batch run, `utils/batch_run.py --run_validation` runs the legacy scripts and `evaluate()` (`--validation_engine legacy|evaluate|both`, `--validation_group_by`) and writes this library's output to `<validation_output_dir>/evaluation`.

In [ ]:
cli = subprocess.run(
    [sys.executable, '-m', 'src.full_stack.backend.utils.validation.with_annotated_dataset', 'evaluate',
     '--predictions', str(EXAMPLES / 'predictions_tidy_example.csv'),
     '--annotations', str(EXAMPLES / 'annotations_table_example.csv'),
     '--id-column', 'participant_id',
     '--annotation-columns', 'root=diagnosis,total_score=total_score',
     '--group-by', 'tier,predictor',
     '--n-boot', '200', '--no-figures',
     '--out', str(OUT / 'cli_example')],
    cwd=REPO, capture_output=True, text=True,
)
print('exit code', cli.returncode)
print(cli.stdout[-1500:] or cli.stderr[-1500:])
print(sorted(p.name for p in (OUT / 'cli_example').iterdir()))

## 10. HPC and Batch Integration Notes

- `hpc/05_submit_batch.sh` forwards `PREDICTION_TYPE` to validation scripts.
- Binary validation expects JSON `TARGETS_FILE`.
- Non-binary validation expects `ANNOTATIONS_JSON`.
- The same annotation-contract logic is shared across local, batch, and HPC workflows.
- `utils/batch_run.py` (a fixed participant list, outside Slurm) forwards the Predictor, routing, decision-model and context-window flags of `main.py`; `--check_config` checks every participant offline (route and budget per record, no model call) and `--preflight_check` runs that check before the cohort.
- `utils/batch_run.py --run_validation` runs the legacy scripts and `evaluate()` (`--validation_engine legacy|evaluate|both`).

Example:

```bash
PREDICTION_TYPE=regression_univariate \
ANNOTATIONS_JSON=~/compass_pipeline/data/__TARGETS__/annotated_targets.json \
bash hpc/05_submit_batch.sh
```

## 11. XAI Scope

XAI methods currently apply only to pure root-level binary classification.

For multiclass/regression/hierarchical validation, outputs remain fully supported and include explicit `xai_status: skipped` metadata.